# Lie Symmetry Analysis of the Viscous Burgers' Equation

This tutorial demonstrates using **`symlie`** to analyze the nonlinear viscous **Burgers' equation**:
$$u_t + u u_x - \nu u_{xx} = 0$$

We investigate:
1. The Fréchet linearization and formal adjoint operator.
2. The 5-dimensional Lie point symmetry group (translations, Galilean boosts, scaling, projective transformations).
3. The Lie bracket commutator algebra.
4. Symmetry reduction and the Cole–Hopf linearization transformation.

In [ ]:
import sympy as sp

from symlie import (
    InfinitesimalGenerator,
    frechet_derivative,
    infinitesimals,
    lie_bracket,
    max_derivative_order,
    verify_generator,
)

sp.init_printing()

x, t = sp.symbols("x t")
nu = sp.symbols("nu", positive=True)
u = sp.Function("u")(x, t)

# Viscous Burgers' Equation
burgers_eq = u.diff(t) + u * u.diff(x) - nu * u.diff(x, 2)
print("PDE Order:", max_derivative_order(burgers_eq, u, (x, t)))
sp.Eq(burgers_eq, 0)

## 1. Fréchet Derivative (Linearization)

The Fréchet derivative of Burgers' equation acting on a perturbation field $Q(x, t)$ is:
$$D_{\text{Burgers}}(Q) = Q_t + u Q_x + u_x Q - \nu Q_{xx}$$

In [ ]:
Q = sp.Function("Q")(x, t)
D_burgers = frechet_derivative(burgers_eq, u, (x, t), Q)
display(D_burgers)

## 2. Lie Point Symmetry Generators

Solving the determining system with `infinitesimals(ansatz_degree=1)` yields the 4 basic symmetries (translations, Galilean boost, and scaling):

In [ ]:
sol = infinitesimals(burgers_eq.subs(nu, 1), u, (x, t), ansatz_degree=1)
print(f"Dimension of degree-1 Lie algebra: {sol.dimension}\n")

# NOTE: these labels are matched positionally to sol.basis; they assume
# a stable solver ordering and are not independently derived from each
# generator's xi/phi content.
labels = [
    "X_1 (Space Translation):",
    "X_2 (Time Translation):",
    "X_3 (Galilean Boost):",
    "X_4 (Scaling / Dilation):",
]

for label, gen in zip(labels, sol.basis):
    is_valid = verify_generator(burgers_eq, u, (x, t), gen)
    print(f"{label}")
    print(f"  xi^x = {gen.xi[0]},  xi^t = {gen.xi[1]},  phi^u = {gen.phi[0]}")
    print(f"  Verified invariant: {is_valid}\n")

Additionally, Burgers' equation admits the **projective symmetry** (degree 2):
$$X_5 = t x \frac{\partial}{\partial x} + t^2 \frac{\partial}{\partial t} + (x - t u) \frac{\partial}{\partial u}$$

In [ ]:
X5_projective = InfinitesimalGenerator(xi=(t * x, t**2), phi=(x - t * u,))
print(
    "X_5 (Projective Generator) Invariant:",
    verify_generator(burgers_eq, u, (x, t), X5_projective),
)

## 3. Lie Bracket Commutator Algebra

We evaluate commutators $[X_i, X_j]$ using `lie_bracket`:

In [ ]:
X1, X2, X3, X4 = sol.basis

print("[X_1, X_4] =", lie_bracket(X1, X4, u, (x, t)))
print("[X_2, X_3] =", lie_bracket(X2, X3, u, (x, t)))
print("[X_2, X_4] =", lie_bracket(X2, X4, u, (x, t)))
print("[X_3, X_4] =", lie_bracket(X3, X4, u, (x, t)))

## 4. The Cole–Hopf Linearization Transformation

The nonlinear Burgers' equation can be mapped to the linear heat equation $v_t = \nu v_{xx}$ via the **Cole–Hopf transformation**:
$$u(x, t) = -2\nu \frac{v_x(x, t)}{v(x, t)}$$

In [ ]:
v = sp.Function("v")(x, t)
cole_hopf = -2 * nu * v.diff(x) / v

# Substitute Cole-Hopf ansatz into Burgers' equation
pde_sub = burgers_eq.subs(u, cole_hopf).doit()
pde_simplified = sp.factor(sp.simplify(pde_sub))

print("Burgers equation under Cole-Hopf transformation:")
display(pde_simplified)
print(
    "Notice the numerator is v * D_x(v_t - nu*v_xx) - v_x * (v_t - nu*v_xx): it vanishes"
    "exactly when v solves the heat equation v_t = nu*v_xx (Cole-Hopf linearization)!"
)